# Шаг 18. Построение агрегированных таблиц

Агрегация нужна для перехода от отдельных строк к сводным показателям. 

Это позволит:
- ответить на вопросы о структуре рынка (кто лидеры, кто аутсайдеры);
- отследить эволюцию ассортимента и качества по десятилетиям;
- увидеть фокусы производителей в разрезе эпох и национальностей;
- оценить, как размер набора связан с его качеством;
- выявить актуальные хиты через адаптированный RFM-анализ;
- оценить стабильность присутствия производителя в нише через адаптированный XYZ-анализ.

Агрегации будут применены для ответа на следующие аналитические вопросы из Шага 2:

**1. ABC-анализ производителей (Структура рынка):**
- Какие производители выпускают наибольшее количество наборов?
- Какие 20% производителей дают 80% всего ассортимента?
- Как распределены производители по категориям A/B/C (доля брендов vs доля ассортимента)?

**2. Динамика рынка по десятилетиям:**
- Как менялось количество выпускаемых наборов с 1950-х годов по настоящее время?
- Как эволюционировало среднее качество наборов (рейтинг) по десятилетиям?

**3. Карта исторических периодов (Эпоха × Национальность):**
- Какие национальности наиболее проработаны в каждой исторической эпохе?

**4. Связь количества фигур и рейтинга:**
- Как средний рейтинг качества зависит от размера набора (мини, стандарт, крупный, сэмплер/диорама)?

**5. Адаптированный RFM-анализ (Актуальные хиты):**
- Для каждой пары (производитель, национальность) оцениваются:
  - **Recency** — сколько лет прошло с последнего релиза (чем меньше, тем лучше);
  - **Frequency** — сколько всего уникальных наборов выпущено (чем больше, тем лучше);
  - **Magnitude** — средний рейтинг качества (чем больше, тем лучше).
- Отбираются только «актуальные хиты»: Recency ≤ 5 лет и Magnitude ≥ 40.

**6. Адаптированный XYZ-анализ (Стабильность присутствия в нише):**
- Насколько стабильно производитель поддерживает конкретную нишу (национальность)?
- Какие производители работают с темой регулярно (категория X), волнообразно (Y) или эпизодически (Z)?

**Адаптация XYZ-анализа:**
Классический XYZ оценивает стабильность *спроса* по временным рядам продаж (например, по месяцам). В данном случае нет данных о продажах, остатках или ежемесячном спросе — есть только факты релизов наборов (`release_year`). Поэтому будет проведена оценка **стабильности присутствия производителя в нише** через метрику `regularity` — долю лет с релизами внутри активного периода производителя.

**Ограничения XYZ-анализа:**
- Метрика имеет смысл только при `kit_count >= 3`. Для 1–2 наборов расчёт некорректен — такие группы помечаются как «Недостаточно данных».
- Это анализ *предложения* (supply), а не *спроса* (demand).
- Метрика не учитывает объём выпусков в каждом году, только факт присутствия.

In [6]:
import pandas as pd
import numpy as np

# Загружаем очищенный датасет с явным указанием типов
df_long = pd.read_csv('df_consolidated_clean.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

# Загружаем данные по наборам для ABC-анализа
df_sets = pd.read_csv('df_sets.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

CURRENT_YEAR = 2026  # Фиксируем текущий год для расчётов

print(f"Загружено строк в df_long: {len(df_long)}")
print(f"Загружено строк в df_sets: {len(df_sets)}")

# =============================================================================
# 1. ABC-АНАЛИЗ ПРОИЗВОДИТЕЛЕЙ (Структура рынка)
# =============================================================================
print("\n" + "="*70)
print("=== 1. ABC-анализ производителей (Структура рынка) ===")
print("="*70)

agg_manufacturers = df_sets.groupby('manufacturer').agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean'),
    avg_figures=('num_figures', 'mean')
).sort_values('kit_count', ascending=False).reset_index()

total_kits = agg_manufacturers['kit_count'].sum()
agg_manufacturers['cumulative_pct'] = (agg_manufacturers['kit_count'].cumsum() / total_kits * 100).round(1)

def get_abc_category(pct):
    if pct <= 80: return 'A'
    elif pct <= 95: return 'B'
    else: return 'C'

agg_manufacturers['ABC_Category'] = agg_manufacturers['cumulative_pct'].apply(get_abc_category)

print("\nТоп-10 производителей по количеству наборов:")
display(agg_manufacturers.head(10))

# Распределение производителей по ABC-категориям
print("\nРаспределение производителей по ABC-категориям:")
abc_distribution = agg_manufacturers['ABC_Category'].value_counts().sort_index()
print(abc_distribution)

# Дополнительно: доля производителей и доля ассортимента в каждой категории
abc_summary = agg_manufacturers.groupby('ABC_Category').agg(
    manufacturers_count=('manufacturer', 'count'),
    total_kits=('kit_count', 'sum')
).reset_index()
abc_summary['share_of_manufacturers_pct'] = (abc_summary['manufacturers_count'] / abc_summary['manufacturers_count'].sum() * 100).round(1)
abc_summary['share_of_kits_pct'] = (abc_summary['total_kits'] / abc_summary['total_kits'].sum() * 100).round(1)

print("\nСтруктура рынка по ABC-категориям:")
display(abc_summary)

agg_manufacturers.to_csv('agg_manufacturers_abc.csv', index=False, encoding='utf-8')
print("✅ Сохранено: 'agg_manufacturers_abc.csv'")

# =============================================================================
# 2. ДИНАМИКА РЫНКА ПО ДЕСЯТИЛЕТИЯМ
# =============================================================================
print("\n" + "="*70)
print("=== 2. Динамика рынка по десятилетиям ===")
print("="*70)

agg_decades = df_long.groupby('decade').agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean')
).reset_index().sort_values('decade')

print("\nДинамика количества наборов и среднего рейтинга по десятилетиям:")
display(agg_decades)

agg_decades.to_csv('agg_decades_dynamics.csv', index=False, encoding='utf-8')
print("✅ Сохранено: 'agg_decades_dynamics.csv'")

# =============================================================================
# 3. КАРТА ИСТОРИЧЕСКИХ ПЕРИОДОВ (Эпоха × Национальность, Топ-5, по хронологии)
# =============================================================================
print("\n" + "="*70)
print("=== 3. Карта исторических периодов (Топ-5 национальностей по эпохам, по хронологии) ===")
print("="*70)

if 'era' in df_long.columns:
    # 1. Исключаем "Не указано"
    df_era_nat = df_long[df_long['nationality'] != 'Не указано'].copy()
    
    # 2. Группируем и считаем количество уникальных наборов
    agg_era_nat = df_era_nat.groupby(['era', 'nationality']).agg(
        kit_count=('id', 'nunique')
    ).reset_index()
    
    # 3. Задаём хронологический порядок эпох (как в Шаге 22)
    era_order = ['Древний мир', 'Средневековье', 'Новое время', 'Новейшее время', 'Современность']
    agg_era_nat['era'] = pd.Categorical(agg_era_nat['era'], categories=era_order, ordered=True)
    
    # 4. Сортируем: сначала по хронологии эпох, затем по убыванию количества наборов
    agg_era_nat = agg_era_nat.sort_values(['era', 'kit_count'], ascending=[True, False])
    
    # 5. Берем только Топ-5 национальностей для каждой эпохи
    agg_era_nat_top5 = agg_era_nat.groupby('era', observed=False).head(5).reset_index(drop=True)
    
    print("\nТоп-5 национальностей по каждой исторической эпохе (в хронологическом порядке):")
    display(agg_era_nat_top5)
    
    # 6. Сохраняем результат
    agg_era_nat_top5.to_csv('agg_era_nationality_map.csv', index=False, encoding='utf-8')
    print("✅ Сохранено: 'agg_era_nationality_map.csv'")
else:
    print("⚠️ Колонка 'era' не найдена в датасете. Проверьте шаги обогащения данных.")

# =============================================================================
# 4. СВЯЗЬ КОЛИЧЕСТВА ФИГУР И РЕЙТИНГА
# =============================================================================
print("\n" + "="*70)
print("=== 4. Связь количества фигур и среднего рейтинга ===")
print("="*70)

def get_figure_category(n):
    if pd.isna(n): return 'Не указано'
    if n <= 15: return 'Мини (1-15)'
    elif n <= 30: return 'Стандарт (16-30)'
    elif n <= 50: return 'Крупный (31-50)'
    else: return 'Сэмплер/Диорама (51+)'

df_long['figure_category'] = df_long['num_figures'].apply(get_figure_category)

agg_figures = df_long.groupby('figure_category').agg(
    kit_count=('id', 'nunique'),
    avg_rating=('aggregate_rating', 'mean')
).reset_index()

# Задаём правильный порядок сортировки для категорий
order = ['Мини (1-15)', 'Стандарт (16-30)', 'Крупный (31-50)', 'Сэмплер/Диорама (51+)', 'Не указано']
agg_figures['sort_order'] = agg_figures['figure_category'].apply(lambda x: order.index(x) if x in order else 99)
agg_figures = agg_figures.sort_values('sort_order').drop(columns='sort_order')

print("\nСредний рейтинг в зависимости от размера набора:")
display(agg_figures)

agg_figures.to_csv('agg_figure_categories.csv', index=False, encoding='utf-8')
print("✅ Сохранено: 'agg_figure_categories.csv'")

# =============================================================================
# 5. АДАПТИРОВАННЫЙ RFM-АНАЛИЗ (Актуальные хиты)
# =============================================================================
print("\n" + "="*70)
print("=== 5. Адаптированный RFM-анализ: Актуальные хиты ===")
print("="*70)

df_rfm = df_long[
    (df_long['release_year'].notna()) &
    (df_long['aggregate_rating'].notna()) &
    (df_long['nationality'] != 'Не указано')
].copy()

agg_rfm = df_rfm.groupby(['manufacturer', 'nationality']).agg(
    Recency=('release_year', lambda x: CURRENT_YEAR - x.max()),
    Frequency=('id', 'nunique'),
    Magnitude=('aggregate_rating', 'mean')
).reset_index()

agg_rfm_hits = agg_rfm.sort_values(['Recency', 'Magnitude'], ascending=[True, False])
agg_rfm_hits = agg_rfm_hits[
    (agg_rfm_hits['Recency'] <= 5) &
    (agg_rfm_hits['Magnitude'] >= 40)
]

print(f"\nНайдено актуальных хитов (Recency ≤ 5 лет, Magnitude ≥ 40): {len(agg_rfm_hits)}")
display(agg_rfm_hits.head(10))

agg_rfm_hits.to_csv('agg_rfm_actual_hits.csv', index=False, encoding='utf-8')
print("✅ Сохранено: 'agg_rfm_actual_hits.csv'")

# =============================================================================
# 6. АДАПТИРОВАННЫЙ XYZ-АНАЛИЗ (Стабильность присутствия в нише)
# =============================================================================
print("\n" + "="*70)
print("=== 6. Адаптированный XYZ-анализ: стабильность присутствия в нише ===")
print("="*70)

df_xyz_base = df_long[
    (df_long['release_year'].notna()) &
    (df_long['nationality'] != 'Не указано')
].copy()

xyz_grouped = df_xyz_base.groupby(['manufacturer', 'nationality']).agg(
    kit_count=('id', 'nunique'),
    first_year=('release_year', 'min'),
    last_year=('release_year', 'max'),
    unique_years=('release_year', 'nunique')
).reset_index()

xyz_grouped['span'] = xyz_grouped['last_year'] - xyz_grouped['first_year'] + 1
xyz_grouped['regularity'] = xyz_grouped['unique_years'] / xyz_grouped['span']
xyz_grouped['has_enough_data'] = xyz_grouped['kit_count'] >= 3

def assign_xyz_category(row):
    if not row['has_enough_data']:
        return 'Недостаточно данных'
    r = row['regularity']
    if r >= 0.5: return 'X'
    elif r >= 0.2: return 'Y'
    else: return 'Z'

xyz_grouped['XYZ_Category'] = xyz_grouped.apply(assign_xyz_category, axis=1)

category_order = {'X': 1, 'Y': 2, 'Z': 3, 'Недостаточно данных': 4}
xyz_grouped['sort_key'] = xyz_grouped['XYZ_Category'].map(category_order)
xyz_grouped = xyz_grouped.sort_values(['sort_key', 'kit_count'], ascending=[True, False]).drop(columns=['sort_key'])

print("\nРаспределение пар (производитель, национальность) по XYZ-категориям:")
print(xyz_grouped['XYZ_Category'].value_counts())

xyz_grouped.to_csv('agg_manufacturers_xyz.csv', index=False, encoding='utf-8')
print("✅ Сохранено: 'agg_manufacturers_xyz.csv'")

# =============================================================================
# ИТОГ ШАГА 18
# =============================================================================
print("\n" + "="*70)
print("ИТОГ ШАГА 18: Все агрегированные таблицы успешно подготовлены")
print("="*70)
print("✅ 'agg_manufacturers_abc.csv'")
print("✅ 'agg_decades_dynamics.csv'")
print("✅ 'agg_era_nationality_map.csv'")
print("✅ 'agg_figure_categories.csv'")
print("✅ 'agg_rfm_actual_hits.csv'")
print("✅ 'agg_manufacturers_xyz.csv'")
print("="*70)

Загружено строк в df_long: 3504
Загружено строк в df_sets: 2840

=== 1. ABC-анализ производителей (Структура рынка) ===

Топ-10 производителей по количеству наборов:


,manufacturer,kit_count,avg_rating,avg_figures,cumulative_pct,ABC_Category
0,Strelets,448,41.849188,37.002262,15.8,A
1,HaT,348,41.454545,35.565625,28.0,A
2,Zvezda,176,45.923077,14.841176,34.2,A
3,RedBox,155,39.945946,30.503268,39.7,A
4,Mars,145,35.446429,33.239669,44.8,A
5,Italeri,120,42.40625,32.295918,49.0,A
6,Caesar,118,45.71,31.034483,53.2,A
7,Preiser,94,46.766667,15.011111,56.5,A
8,Revell,84,44.882353,39.318841,59.4,A
9,Linear-A,79,42.859155,24.74359,62.2,A



Распределение производителей по ABC-категориям:
ABC_Category
A    18
B    18
C    25
Name: count, dtype: int64

Структура рынка по ABC-категориям:


,ABC_Category,manufacturers_count,total_kits,share_of_manufacturers_pct,share_of_kits_pct
0,A,18,2240,29.5,78.9
1,B,18,453,29.5,16.0
2,C,25,147,41.0,5.2


✅ Сохранено: 'agg_manufacturers_abc.csv'

=== 2. Динамика рынка по десятилетиям ===

Динамика количества наборов и среднего рейтинга по десятилетиям:


...

✅ Сохранено: 'agg_era_nationality_map.csv'

=== 4. Связь количества фигур и среднего рейтинга ===

Средний рейтинг в зависимости от размера набора:


,figure_category,kit_count,avg_rating
1,Мини (1-15),790,40.507022
3,Стандарт (16-30),436,40.690045
0,Крупный (31-50),1088,41.322115
4,Сэмплер/Диорама (51+),179,42.325581
2,Не указано,347,27.577465


✅ Сохранено: 'agg_figure_categories.csv'

=== 5. Адаптированный RFM-анализ: Актуальные хиты ===

Найдено актуальных хитов (Recency ≤ 5 лет, Magnitude ≥ 40): 74


,manufacturer,nationality,Recency,Frequency,Magnitude
292,Linear-A,Macedonian,0,1,48.0
285,Linear-A,Goth,0,1,45.0
301,Linear-A,Thracian,0,1,45.0
283,Linear-A,French,0,5,44.4
549,Ultima Ratio,French,0,4,43.5
291,Linear-A,Italian,0,37,42.648649
180,HaT,Hessian,0,3,42.0
297,Linear-A,Romanian,0,2,42.0
501,Strelets,French,0,86,41.569767
532,Strelets,USA,0,36,41.083333


✅ Сохранено: 'agg_rfm_actual_hits.csv'

=== 6. Адаптированный XYZ-анализ: стабильность присутствия в нише ===

Распределение пар (производитель, национальность) по XYZ-категориям:
XYZ_Category
Недостаточно данных    422
X                      122
Y                      109
Z                       14
Name: count, dtype: int64
✅ Сохранено: 'agg_manufacturers_xyz.csv'

ИТОГ ШАГА 18: Все агрегированные таблицы успешно подготовлены
✅ 'agg_manufacturers_abc.csv'
✅ 'agg_decades_dynamics.csv'
✅ 'agg_era_nationality_map.csv'
✅ 'agg_figure_categories.csv'
✅ 'agg_rfm_actual_hits.csv'
✅ 'agg_manufacturers_xyz.csv'


## Результат шага
Подготовлены 6 чистых, структурированных CSV-файлов с агрегированными показателями. Эти файлы являются источником данных (Data Source) для подключения к BI-системе, так как они уже сгруппированы, очищены и содержат рассчитанные метрики (накопительный процент, RFM-метрики, категории).